# Multiclass Classification with SKaiNET

 * Teaching the basics of multiclass classification, using SKaiNET as the implementation engine.
 * Most introductions to machine learning start with **binary** classification: spam or not-spam,fraud or not-fraud.

One question, two answers. But the moment you leave the textbook, the worldstops being binary. Which of six products is a customer about to buy? Which of forty-twointents did the user just express? Which of three species is this flower?That is **multiclass classification**: picking exactly one label out of *N* mutually exclusiveoptions. This notebook builds one end to end — the maths, the representation choices, thetraining loop, and the evaluation — in Kotlin, on the JVM, with[SKaiNET](https://github.com/SKaiNET-developers/SKaiNET).

## What you will learn

| Section | Concept ||---|---|
| 1 | What separates multiclass from binary and multi-label |
| 2–3 | The Iris problem, and why labels become **one-hot vectors** |
| 4 | Wrapping raw data in SKaiNET's `Dataset` so the framework can batch it |
| 5 | **Stratified** train/test splitting with the built-in split DSL |
| 6 | Feature standardisation, and why statistics come from the training set only |
| 7 | **Softmax**, and the logits-vs-probabilities trap that bites nearly everyone |
| 8–9 | Cross-entropy loss and the training DSL |
| 10 | Accuracy, and why a confusion matrix tells you more |
| 11 | Making a single prediction |

## What you need

A Kotlin Jupyter kernel and an internet connection for the first cell. No GPU, no Python,no dataset download — the 150 rows of Iris are embedded below. The whole notebook trains ina couple of seconds on a laptop CPU.> **A note on scope.** This notebook is written against **released SKaiNET 0.40.1**, so> everything here runs today. In a few places we write helper code by hand that arguably> belongs *inside* the library — section 4 is the big one. Those spots are flagged with a> **Room for improvement** callout, and the companion issue proposes fixing them.

In [1]:
@file:DependsOn("sk.ainet.core:skainet-lang-core-jvm:0.40.1")
@file:DependsOn("sk.ainet.core:skainet-backend-cpu-jvm:0.40.1")
@file:DependsOn("sk.ainet.core:skainet-compile-dag-jvm:0.40.1")
@file:DependsOn("sk.ainet.core:skainet-data-api-jvm:0.40.1")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core-jvm:1.10.2")

Four SKaiNET modules, and each one earns its place:- **`skainet-lang-core`** — tensors, layers, losses, optimizers, metrics, and the `sequential { }`  and `training { }` DSLs. The bulk of what we use.- **`skainet-backend-cpu`** — an actual implementation of the tensor operations. SKaiNET separates  *what* to compute from *where* it runs; without a backend you can describe a network but not run it.- **`skainet-compile-dag`** — the gradient tape. Autograd lives here, so this is what makes  training (as opposed to inference) possible.- **`skainet-data-api`** — the `Dataset` / `DataBatch` abstractions we build on in section 4.

---# 1. What is multiclass classification?Three problem shapes are easy to confuse. The difference is entirely in the **structure of theanswer**, and that structure determines your output layer, your loss function, and your metrics.| | Question | Answer shape | Output layer | Typical loss ||---|---|---|---|---|| **Binary** | Is this spam? | one of 2 | 1 unit + sigmoid | binary cross-entropy || **Multiclass** | Which *one* of 3 species? | exactly 1 of N | N units + softmax | categorical cross-entropy || **Multi-label** | Which topics apply? | any subset of N | N units + sigmoid | binary cross-entropy per label |The word doing the heavy lifting in the multiclass row is **exactly**. The classes are *mutuallyexclusive* and *exhaustive*: every flower is one of the three species, and no flower is two ofthem. That single assumption is what licenses softmax, because softmax forces the outputs to sumto 1 — it models a probability distribution *over* the classes, where raising one scorenecessarily lowers the others.If your classes are not mutually exclusive (an article can be about both sport and politics),softmax is the wrong tool and the competition it creates between classes will actively hurt you.Use N independent sigmoids instead.**Our task.** Given four physical measurements of an iris flower, predict which of three speciesit is. Mutually exclusive, exhaustive, three classes. Textbook multiclass.

---# 2. The data: IrisThe Iris dataset was published by the statistician Ronald Fisher in 1936 and has been the"hello, world" of classification ever since. 150 flowers, 50 from each of three species, withfour measurements apiece:- sepal length (cm)- sepal width (cm)- petal length (cm)- petal width (cm)It is a good teaching set for an honest reason and a slightly dishonest one. The honest reason:it is genuinely multiclass, perfectly balanced, and small enough to reason about by hand. Thedishonest one: it is *easy*. One species (*setosa*) is linearly separable from the other two, andthe remaining pair are nearly so. Expect high accuracy, and do not read too much into it — a90%+ score here says very little about how a model will behave on real, messy data.We embed the 150 rows directly so the notebook is fully self-contained.

In [2]:
val IRIS_CSV = """
5.1,3.5,1.4,0.2,Iris-setosa
4.9,3.0,1.4,0.2,Iris-setosa
4.7,3.2,1.3,0.2,Iris-setosa
4.6,3.1,1.5,0.2,Iris-setosa
5.0,3.6,1.4,0.2,Iris-setosa
5.4,3.9,1.7,0.4,Iris-setosa
4.6,3.4,1.4,0.3,Iris-setosa
5.0,3.4,1.5,0.2,Iris-setosa
4.4,2.9,1.4,0.2,Iris-setosa
4.9,3.1,1.5,0.1,Iris-setosa
5.4,3.7,1.5,0.2,Iris-setosa
4.8,3.4,1.6,0.2,Iris-setosa
4.8,3.0,1.4,0.1,Iris-setosa
4.3,3.0,1.1,0.1,Iris-setosa
5.8,4.0,1.2,0.2,Iris-setosa
5.7,4.4,1.5,0.4,Iris-setosa
5.4,3.9,1.3,0.4,Iris-setosa
5.1,3.5,1.4,0.3,Iris-setosa
5.7,3.8,1.7,0.3,Iris-setosa
5.1,3.8,1.5,0.3,Iris-setosa
5.4,3.4,1.7,0.2,Iris-setosa
5.1,3.7,1.5,0.4,Iris-setosa
4.6,3.6,1.0,0.2,Iris-setosa
5.1,3.3,1.7,0.5,Iris-setosa
4.8,3.4,1.9,0.2,Iris-setosa
5.0,3.0,1.6,0.2,Iris-setosa
5.0,3.4,1.6,0.4,Iris-setosa
5.2,3.5,1.5,0.2,Iris-setosa
5.2,3.4,1.4,0.2,Iris-setosa
4.7,3.2,1.6,0.2,Iris-setosa
4.8,3.1,1.6,0.2,Iris-setosa
5.4,3.4,1.5,0.4,Iris-setosa
5.2,4.1,1.5,0.1,Iris-setosa
5.5,4.2,1.4,0.2,Iris-setosa
4.9,3.1,1.5,0.1,Iris-setosa
5.0,3.2,1.2,0.2,Iris-setosa
5.5,3.5,1.3,0.2,Iris-setosa
4.9,3.1,1.5,0.1,Iris-setosa
4.4,3.0,1.3,0.2,Iris-setosa
5.1,3.4,1.5,0.2,Iris-setosa
5.0,3.5,1.3,0.3,Iris-setosa
4.5,2.3,1.3,0.3,Iris-setosa
4.4,3.2,1.3,0.2,Iris-setosa
5.0,3.5,1.6,0.6,Iris-setosa
5.1,3.8,1.9,0.4,Iris-setosa
4.8,3.0,1.4,0.3,Iris-setosa
5.1,3.8,1.6,0.2,Iris-setosa
4.6,3.2,1.4,0.2,Iris-setosa
5.3,3.7,1.5,0.2,Iris-setosa
5.0,3.3,1.4,0.2,Iris-setosa
7.0,3.2,4.7,1.4,Iris-versicolor
6.4,3.2,4.5,1.5,Iris-versicolor
6.9,3.1,4.9,1.5,Iris-versicolor
5.5,2.3,4.0,1.3,Iris-versicolor
6.5,2.8,4.6,1.5,Iris-versicolor
5.7,2.8,4.5,1.3,Iris-versicolor
6.3,3.3,4.7,1.6,Iris-versicolor
4.9,2.4,3.3,1.0,Iris-versicolor
6.6,2.9,4.6,1.3,Iris-versicolor
5.2,2.7,3.9,1.4,Iris-versicolor
5.0,2.0,3.5,1.0,Iris-versicolor
5.9,3.0,4.2,1.5,Iris-versicolor
6.0,2.2,4.0,1.0,Iris-versicolor
6.1,2.9,4.7,1.4,Iris-versicolor
5.6,2.9,3.6,1.3,Iris-versicolor
6.7,3.1,4.4,1.4,Iris-versicolor
5.6,3.0,4.5,1.5,Iris-versicolor
5.8,2.7,4.1,1.0,Iris-versicolor
6.2,2.2,4.5,1.5,Iris-versicolor
5.6,2.5,3.9,1.1,Iris-versicolor
5.9,3.2,4.8,1.8,Iris-versicolor
6.1,2.8,4.0,1.3,Iris-versicolor
6.3,2.5,4.9,1.5,Iris-versicolor
6.1,2.8,4.7,1.2,Iris-versicolor
6.4,2.9,4.3,1.3,Iris-versicolor
6.6,3.0,4.4,1.4,Iris-versicolor
6.8,2.8,4.8,1.4,Iris-versicolor
6.7,3.0,5.0,1.7,Iris-versicolor
6.0,2.9,4.5,1.5,Iris-versicolor
5.7,2.6,3.5,1.0,Iris-versicolor
5.5,2.4,3.8,1.1,Iris-versicolor
5.5,2.4,3.7,1.0,Iris-versicolor
5.8,2.7,3.9,1.2,Iris-versicolor
6.0,2.7,5.1,1.6,Iris-versicolor
5.4,3.0,4.5,1.5,Iris-versicolor
6.0,3.4,4.5,1.6,Iris-versicolor
6.7,3.1,4.7,1.5,Iris-versicolor
6.3,2.3,4.4,1.3,Iris-versicolor
5.6,3.0,4.1,1.3,Iris-versicolor
5.5,2.5,4.0,1.3,Iris-versicolor
5.5,2.6,4.4,1.2,Iris-versicolor
6.1,3.0,4.6,1.4,Iris-versicolor
5.8,2.6,4.0,1.2,Iris-versicolor
5.0,2.3,3.3,1.0,Iris-versicolor
5.6,2.7,4.2,1.3,Iris-versicolor
5.7,3.0,4.2,1.2,Iris-versicolor
5.7,2.9,4.2,1.3,Iris-versicolor
6.2,2.9,4.3,1.3,Iris-versicolor
5.1,2.5,3.0,1.1,Iris-versicolor
5.7,2.8,4.1,1.3,Iris-versicolor
6.3,3.3,6.0,2.5,Iris-virginica
5.8,2.7,5.1,1.9,Iris-virginica
7.1,3.0,5.9,2.1,Iris-virginica
6.3,2.9,5.6,1.8,Iris-virginica
6.5,3.0,5.8,2.2,Iris-virginica
7.6,3.0,6.6,2.1,Iris-virginica
4.9,2.5,4.5,1.7,Iris-virginica
7.3,2.9,6.3,1.8,Iris-virginica
6.7,2.5,5.8,1.8,Iris-virginica
7.2,3.6,6.1,2.5,Iris-virginica
6.5,3.2,5.1,2.0,Iris-virginica
6.4,2.7,5.3,1.9,Iris-virginica
6.8,3.0,5.5,2.1,Iris-virginica
5.7,2.5,5.0,2.0,Iris-virginica
5.8,2.8,5.1,2.4,Iris-virginica
6.4,3.2,5.3,2.3,Iris-virginica
6.5,3.0,5.5,1.8,Iris-virginica
7.7,3.8,6.7,2.2,Iris-virginica
7.7,2.6,6.9,2.3,Iris-virginica
6.0,2.2,5.0,1.5,Iris-virginica
6.9,3.2,5.7,2.3,Iris-virginica
5.6,2.8,4.9,2.0,Iris-virginica
7.7,2.8,6.7,2.0,Iris-virginica
6.3,2.7,4.9,1.8,Iris-virginica
6.7,3.3,5.7,2.1,Iris-virginica
7.2,3.2,6.0,1.8,Iris-virginica
6.2,2.8,4.8,1.8,Iris-virginica
6.1,3.0,4.9,1.8,Iris-virginica
6.4,2.8,5.6,2.1,Iris-virginica
7.2,3.0,5.8,1.6,Iris-virginica
7.4,2.8,6.1,1.9,Iris-virginica
7.9,3.8,6.4,2.0,Iris-virginica
6.4,2.8,5.6,2.2,Iris-virginica
6.3,2.8,5.1,1.5,Iris-virginica
6.1,2.6,5.6,1.4,Iris-virginica
7.7,3.0,6.1,2.3,Iris-virginica
6.3,3.4,5.6,2.4,Iris-virginica
6.4,3.1,5.5,1.8,Iris-virginica
6.0,3.0,4.8,1.8,Iris-virginica
6.9,3.1,5.4,2.1,Iris-virginica
6.7,3.1,5.6,2.4,Iris-virginica
6.9,3.1,5.1,2.3,Iris-virginica
5.8,2.7,5.1,1.9,Iris-virginica
6.8,3.2,5.9,2.3,Iris-virginica
6.7,3.3,5.7,2.5,Iris-virginica
6.7,3.0,5.2,2.3,Iris-virginica
6.3,2.5,5.0,1.9,Iris-virginica
6.5,3.0,5.2,2.0,Iris-virginica
6.2,3.4,5.4,2.3,Iris-virginica
5.9,3.0,5.1,1.8,Iris-virginica
""".trimIndent()

val CLASS_NAMES = listOf("Iris-setosa", "Iris-versicolor", "Iris-virginica")
val FEATURE_NAMES = listOf("sepalLength", "sepalWidth", "petalLength", "petalWidth")

val NUM_FEATURES = FEATURE_NAMES.size   // 4
val NUM_CLASSES = CLASS_NAMES.size      // 3

println("Rows in CSV: " + IRIS_CSV.lines().count { it.isNotBlank() })

Rows in CSV: 150


Now parse it. Each row becomes a `FloatArray` of four features plus an `Int` class index in`0..2`. Note the class index, not the class *name* — models consume numbers, and mapping thename to a stable integer position is the first thing every classification pipeline does.

In [3]:
data class IrisSample(val features: FloatArray, val label: Int)

val samples: List<IrisSample> = IRIS_CSV.lines()
    .filter { it.isNotBlank() }
    .map { line ->
        val cols = line.split(",")
        IrisSample(
            features = FloatArray(NUM_FEATURES) { i -> cols[i].trim().toFloat() },
            label = CLASS_NAMES.indexOf(cols[NUM_FEATURES].trim())
                .also { require(it >= 0) { "Unknown species in row: $line" } }
        )
    }

println("Parsed ${samples.size} samples")
println()
samples.groupBy { it.label }.toSortedMap().forEach { (label, group) ->
    println("  ${CLASS_NAMES[label].padEnd(16)} -> ${group.size} samples")
}
println()
println("First sample: features=${samples[0].features.toList()}, label=${samples[0].label} (${CLASS_NAMES[samples[0].label]})")

Parsed 150 samples

  Iris-setosa      -> 50 samples
  Iris-versicolor  -> 50 samples
  Iris-virginica   -> 50 samples

First sample: features=[5.1, 3.5, 1.4, 0.2], label=0 (Iris-setosa)


---# 3. Representing the labels: one-hot encodingWe have labels as integers: `0`, `1`, `2`. Why not feed those to the network directly?Because **integers imply an ordering that does not exist**. If the model learns to output asingle number and we train it to emit `0` for setosa, `1` for versicolor and `2` for virginica,we have quietly told it that versicolor sits "between" the other two, and that setosa andvirginica are "further apart" than setosa and versicolor. That is nonsense — these are names,not quantities. A model trained this way will hedge toward the middle class, because predicting`1` is never catastrophically wrong under a numeric loss.The fix is **one-hot encoding**: represent each label as a vector of length *N* that is 1 at thetrue class position and 0 everywhere else.```Iris-setosa      (class 0)  ->  [1, 0, 0]Iris-versicolor  (class 1)  ->  [0, 1, 0]Iris-virginica   (class 2)  ->  [0, 0, 1]```Now all three labels are equidistant from each other. No accidental ordering, no implied metric.There is a second, deeper reason. A one-hot vector *is* a probability distribution — one thathappens to be completely certain. The network will also output a distribution over three classes.Having both sides in the same format is what lets us compare them with cross-entropy in section 8.> **SKaiNET note.** SKaiNET's `CategoricalCrossEntropyLoss` accepts one-hot (soft) targets *or*> raw `Int32` class indices — the sibling `SparseCategoricalCrossEntropyLoss` name signals the> latter. We use one-hot here because it makes the "distribution vs distribution" idea concrete,> and because the `Accuracy` metric handles both identically.

In [4]:
fun oneHot(classIndex: Int, numClasses: Int = NUM_CLASSES): FloatArray =
    FloatArray(numClasses) { if (it == classIndex) 1.0f else 0.0f }

CLASS_NAMES.indices.forEach { i ->
    println("${CLASS_NAMES[i].padEnd(16)} (class $i) -> ${oneHot(i).toList()}")
}

Iris-setosa      (class 0) -> [1.0, 0.0, 0.0]
Iris-versicolor  (class 1) -> [0.0, 1.0, 0.0]
Iris-virginica   (class 2) -> [0.0, 0.0, 1.0]


---# 4. Wrapping the data in SKaiNET's `Dataset`We could stop here, hold the samples in a `List`, and hand-roll our own batching and splittingloops. That is exactly what a first draft usually does — and it is exactly what we should avoid,because SKaiNET's `Dataset` abstraction already provides all of it.`Dataset<X, Y>` is an abstract class in `skainet-data-api`. Implement a handful of members andyou inherit, for free:| Inherited capability | What it gives you ||---|---|| `split(ratio, seed, stratified)` | reproducible, class-balanced train/test splits || `shuffle(seed)` | deterministic shuffling || `batches(batchSize, shuffle, seed)` | a cold `Flow<DataBatch>` of tensor batches || `epochs(n, batchSize, ...)` | the same, across *n* epochs, reshuffled each time || `filter`, `mapX`, `mapY` | lazy dataset views |The members we must supply are `xSize`, `getX`, `getY`, `shuffle()`, `split(ratio)` and`createDataBatch(...)` — that last one being the interesting one, since it's where raw Kotlinarrays become SKaiNET tensors.One subtlety worth understanding, because it is easy to get wrong: `split` and `shuffle` returnan **index view** over the original dataset rather than copying data. When you then ask that viewfor a batch, it requests a set of *non-contiguous* indices from the underlying dataset. The baseclass can only serve contiguous ranges, so it throws unless you also override`createIndexedDataBatch`. Override both and shuffled, split, filtered views all just work.

In [5]:
import sk.ainet.context.DefaultDataExecutionContext
import sk.ainet.context.ExecutionContext
import sk.ainet.data.DataBatch
import sk.ainet.data.Dataset
import sk.ainet.lang.tensor.Shape
import sk.ainet.lang.tensor.Tensor
import sk.ainet.lang.types.DType
import sk.ainet.lang.types.FP32
import kotlin.math.min
import kotlin.random.Random

/**
 * A minimal tabular Dataset over in-memory Iris samples.
 *
 *  x -> Tensor FP32 [batch, 4]   the four measurements
 *  y -> Tensor FP32 [batch, 3]   one-hot species
 */
class IrisDataset(
    private val items: List<IrisSample>,
    private val ctx: ExecutionContext = DefaultDataExecutionContext()
) : Dataset<FloatArray, Int>() {

    override val inputShape: Shape get() = Shape(NUM_FEATURES)
    override val outputShape: Shape get() = Shape(NUM_CLASSES)

    override val xSize: Int get() = items.size

    override fun getX(idx: Int): FloatArray = items[idx].features
    override fun getY(idx: Int): Int = items[idx].label

    override fun shuffle(): Dataset<FloatArray, Int> =
        IrisDataset(items.shuffled(Random.Default), ctx)

    override fun split(splitRatio: Double): Pair<Dataset<FloatArray, Int>, Dataset<FloatArray, Int>> {
        require(splitRatio > 0.0 && splitRatio < 1.0) { "splitRatio must be in (0,1)" }
        val at = (items.size * splitRatio).toInt()
        return IrisDataset(items.subList(0, at), ctx) to IrisDataset(items.subList(at, items.size), ctx)
    }

    // Contiguous batches (the plain, unshuffled path).
    override fun <T : DType, V> createDataBatch(batchStart: Int, batchLength: Int): DataBatch<T, V> {
        val len = min(batchLength, xSize - batchStart)
        return batchFor(IntArray(len) { batchStart + it })
    }

    // Arbitrary indices — required for shuffled / split / filtered views.
    override fun <T : DType, V> createIndexedDataBatch(indices: IntArray): DataBatch<T, V> =
        batchFor(indices)

    @Suppress("UNCHECKED_CAST")
    private fun <T : DType, V> batchFor(indices: IntArray): DataBatch<T, V> {
        val n = indices.size

        val xData = FloatArray(n * NUM_FEATURES)
        val yData = FloatArray(n * NUM_CLASSES)
        indices.forEachIndexed { row, srcIdx ->
            val sample = items[srcIdx]
            sample.features.copyInto(xData, destinationOffset = row * NUM_FEATURES)
            oneHot(sample.label).copyInto(yData, destinationOffset = row * NUM_CLASSES)
        }

        val x: Tensor<FP32, Float> = ctx.fromFloatArray(Shape(n, NUM_FEATURES), FP32::class, xData)
        val y: Tensor<FP32, Float> = ctx.fromFloatArray(Shape(n, NUM_CLASSES), FP32::class, yData)

        return DataBatch(
            x = arrayOf(x) as Array<Tensor<T, V>>,
            y = y as Tensor<T, V>,
            indices = indices
        )
    }
}

> ### 🔧 Room for improvement>> **This entire cell is boilerplate that should not be in a notebook.** SKaiNET ships ready-made> providers for MNIST, Fashion-MNIST and CIFAR-10 in `skainet-data-simple` — you write> `MNIST.loadTrain()` and you are done. There is no equivalent for Iris, and more importantly no> general *tabular* bridge: `skainet-data-source` will happily parse a CSV into a `RawDataset`> (rows as `Map<String, String>` plus a schema), but nothing turns a `RawDataset` into a> `Dataset` that emits tensor batches. That missing adapter is why this cell exists.>> The companion issue proposes both pieces. Once they land, everything above collapses to:>> ```kotlin> val dataset = Iris.load()   // suspend, cached, one line> ```>> We keep the hand-written version for now so this notebook runs on released 0.40.1.

In [6]:
val dataset = IrisDataset(samples)

println("Dataset size:  ${dataset.size}")
println("Input shape:   ${dataset.inputShape}")
println("Output shape:  ${dataset.outputShape}")

Dataset size:  150
Input shape:   Shape: Dimensions = [4], Size (Volume) = 4
Output shape:  Shape: Dimensions = [3], Size (Volume) = 3


---# 5. Splitting the data — and why *stratified* mattersA model that is evaluated on the data it trained on tells you nothing. It might have learned theunderlying pattern, or it might have memorised 150 rows; both look identical on the training set.So we hold part of the data back and only ever look at it at the end.The naive split is "shuffle, then take the first 80%". SKaiNET's `Dataset.split` does better —it takes a `seed` for reproducibility and a `stratified` flag:```kotlinval (train, test) = dataset.split(0.8, seed = 42L, stratified = true)```**Stratified** means the split preserves the class proportions in both halves. With 150 samplesand three balanced classes this sounds pedantic, but run the numbers on a random split: the testset is 30 flowers, and it is entirely plausible to draw 15 setosa, 12 versicolor and 3 virginica.Your virginica accuracy is then measured on three examples, and a single mistake moves it by 33percentage points. On imbalanced real-world data — fraud detection, rare-disease diagnosis — anon-stratified split can produce a test set containing *zero* examples of the minority class,at which point your evaluation is silently meaningless.Stratification costs nothing and removes an entire category of confusing results. Use it bydefault.> **How it works internally.** SKaiNET buckets sample indices by the value of `getY(idx)`, shuffles> within each bucket using the seed, splits each bucket at the ratio, then shuffles the two> assembled index lists. This is precisely why our `getY` returns the `Int` class index and not> the one-hot `FloatArray` — buckets are keyed by `Y`, and arrays compare by identity, so a> `FloatArray` label would put every single sample in its own bucket and silently defeat> stratification.

In [7]:
val (trainSet, testSet) = dataset.split(0.8, seed = 42L, stratified = true)

fun Dataset<FloatArray, Int>.classCounts(): Map<Int, Int> =
    (0 until size).groupingBy { getY(it) }.eachCount().toSortedMap()

println("Train: ${trainSet.size} samples   ${trainSet.classCounts()}")
println("Test:  ${testSet.size} samples   ${testSet.classCounts()}")
println()
println("Class balance preserved in both halves:")
CLASS_NAMES.indices.forEach { c ->
    val tr = trainSet.classCounts()[c] ?: 0
    val te = testSet.classCounts()[c] ?: 0
    println("  ${CLASS_NAMES[c].padEnd(16)} train=$tr  test=$te")
}

Train: 120 samples   {0=40, 1=40, 2=40}
Test:  30 samples   {0=10, 1=10, 2=10}

Class balance preserved in both halves:
  Iris-setosa      train=40  test=10
  Iris-versicolor  train=40  test=10
  Iris-virginica   train=40  test=10


---# 6. Standardising the featuresLook at the raw measurements: sepal length runs roughly 4.3–7.9 cm, while petal width runs0.1–2.5 cm. Same units, but one feature spans a range about four times wider than the other.Neural networks care about this. Every input feeds through a weighted sum, so a feature with alarger numeric range produces larger gradients and effectively dominates the early updates —not because it is more *informative*, but because it is bigger. Training becomes slower and moresensitive to the learning rate.**Standardisation** puts every feature on the same footing by rescaling to zero mean and unitvariance:$$z = \frac{x - \mu}{\sigma}$$The critical detail — and one of the most common bugs in applied ML — is **where μ and σ comefrom**. They must be computed on the *training set only*, then applied unchanged to the test set.If you standardise using statistics from the full dataset, information about the test set(its mean, its spread) leaks into the training process. Your test score then flatters the model,and the gap only shows up in production. The rule generalises: **any** preprocessing that *learns*something from data — scaling, imputation, vocabulary building, PCA — must learn it from thetraining split alone.

In [8]:
import kotlin.math.sqrt

data class Standardizer(val mean: FloatArray, val std: FloatArray) {
    fun apply(x: FloatArray): FloatArray =
        FloatArray(x.size) { i -> (x[i] - mean[i]) / std[i] }

    companion object {
        /** Fit on TRAINING data only. */
        fun fit(data: Dataset<FloatArray, Int>): Standardizer {
            val n = data.size
            val d = data.getX(0).size

            val mean = FloatArray(d)
            for (i in 0 until n) {
                val x = data.getX(i)
                for (j in 0 until d) mean[j] += x[j]
            }
            for (j in 0 until d) mean[j] /= n

            val std = FloatArray(d)
            for (i in 0 until n) {
                val x = data.getX(i)
                for (j in 0 until d) {
                    val diff = x[j] - mean[j]
                    std[j] += diff * diff
                }
            }
            // Guard against a constant feature producing a divide-by-zero.
            for (j in 0 until d) std[j] = maxOf(sqrt(std[j] / n), 1e-8f)

            return Standardizer(mean, std)
        }
    }
}

val scaler = Standardizer.fit(trainSet)

FEATURE_NAMES.forEachIndexed { i, name ->
    println("${name.padEnd(14)} mean=${"%.3f".format(scaler.mean[i])}  std=${"%.3f".format(scaler.std[i])}")
}

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[8], line 18, column 38: No set method providing array access

Now apply the fitted scaler to both halves. We rebuild `IrisDataset` instances rather than using`Dataset.mapX`, because a `MappedDataset` view deliberately refuses to produce tensor batches(it has no way to know how to tensorize the transformed type) — and tensor batches are exactlywhat we need next.

In [9]:
fun Dataset<FloatArray, Int>.standardized(scaler: Standardizer): IrisDataset =
    IrisDataset((0 until size).map { IrisSample(scaler.apply(getX(it)), getY(it)) })

val trainScaled = trainSet.standardized(scaler)
val testScaled = testSet.standardized(scaler)

println("Before: ${trainSet.getX(0).toList().map { "%.2f".format(it) }}")
println("After:  ${trainScaled.getX(0).toList().map { "%.2f".format(it) }}")

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[9], line 1, column 51: Unresolved reference: Standardizer
at Cell In[9], line 4, column 41: Unresolved reference: scaler
at Cell In[9], line 5, column 39: Unresolved reference: scaler

---# 7. The model, and the softmax trapOur network is deliberately small — four inputs, one hidden layer of eight units, three outputs:```[4 features] -> Dense(8) -> ReLU -> Dense(3) -> [3 scores]```The hidden layer with a ReLU non-linearity is what lets the model draw curved decision boundariesrather than straight lines. Three output units, one per class, is the multiclass signature: theoutput layer width *is* the number of classes.## SoftmaxThe three final numbers are unbounded reals — maybe `[2.1, -0.4, 0.8]`. They are called**logits**, and they are not probabilities. Softmax converts them into one:$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$Exponentiate each score (making everything positive), then divide by the total (making them sumto 1). Applied to `[2.1, -0.4, 0.8]` this gives roughly `[0.70, 0.06, 0.24]` — a genuinedistribution: 70% confident it is class 0.Note the *soft* in the name. It does not just pick the winner; it preserves the relativeconfidence, which is what makes the function differentiable and therefore trainable.## ⚠️ The trap: do not apply softmax twiceHere is the mistake almost everyone makes at least once, and it is worth stating loudly becauseit produces a model that *appears to work*.SKaiNET's `CategoricalCrossEntropyLoss` **applies log-softmax internally**. Its own documentationsays so explicitly: *"The loss applies log-softmax internally, so do NOT apply softmax to yourmodel's output before passing to this loss."*So if you finish your network with a softmax layer *and* use this loss, the model softmaxes asoftmax. The output is still monotonic, so `argmax` still picks the same class and accuracy stilllooks fine — which is why the bug survives code review. But the second softmax squashes analready-flat distribution flatter still, the gradients shrink, and the model trains far moreslowly than it should. On an easy dataset like Iris you may never notice. On a hard one you willlose accuracy and blame the architecture.**The rule:** during *training*, the network outputs **raw logits**. Softmax is applied only at*inference*, when you want human-readable probabilities. We follow that rule below, and you willsee softmax reappear in section 11 where it belongs.

In [10]:
import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.context.Phase
import sk.ainet.lang.graph.DefaultGradientTape
import sk.ainet.lang.graph.DefaultGraphExecutionContext
import sk.ainet.lang.nn.dsl.sequential
import sk.ainet.lang.tensor.relu
import kotlin.math.sqrt

// The eager CPU backend: where the arithmetic actually happens.
val baseCtx = DirectCpuExecutionContext()

// A recording context layered on top — it taps every op onto a gradient tape
// so backpropagation can replay them. Phase.TRAIN also switches training-only
// layers (dropout, batch-norm) into training behaviour.
val trainCtx = DefaultGraphExecutionContext(
    baseOps = baseCtx.ops,
    phase = Phase.TRAIN,
    createTapeFactory = { _ -> DefaultGradientTape() }
)

val model = sequential<FP32, Float>(trainCtx) {
    input(NUM_FEATURES)

    dense(8, "hidden") {
        // He initialisation: std = sqrt(2 / fanIn), the standard choice for ReLU.
        // `shape` here is the weight shape [outputDim, inputDim], so shape[1] is fanIn.
        weights { randn(std = sqrt(2.0f / shape[1])) }
        bias { zeros() }
        activation = { it.relu() }
    }

    // Output layer: NO activation. Raw logits, straight into the loss.
    dense(NUM_CLASSES, "output") {
        weights { randn(std = sqrt(2.0f / shape[1])) }
        bias { zeros() }
    }
}

println(model)
println()
println("Trainable parameter tensors: ${model.trainableParameters().size}")

[SKaiNET] Using standard CPU operations (Vector API not available)
sk.ainet.lang.nn.topology.MLP@69a674e9

Trainable parameter tensors: 4


Two things in that DSL block are worth calling out, because they replace code you would otherwisewrite by hand.**Activation inside `dense`, not after it.** `DENSE` exposes an `activation` property, so`activation = { it.relu() }` attaches the non-linearity to the layer that owns it. The alternative— a separate top-level `activation { it.relu() }` entry — works identically but scatters a layer'sdefinition across two DSL entries.**Initialisation via the weights scope.** Inside `weights { }` you are in a `WeightsScope` withthe target `shape` in scope and factory methods (`randn`, `uniform`, `zeros`, `ones`, `init`)available. Because `shape` is there, fan-in-dependent schemes like He initialisation are aone-liner. Weight initialisation matters more than beginners expect: too large and activationssaturate, too small and the signal dies as it propagates. He initialisation keeps the varianceof activations roughly stable across a ReLU network.

---# 8. The loss: categorical cross-entropyThe model outputs a predicted distribution; the label is a true (one-hot) distribution.**Cross-entropy** measures how far apart two distributions are:$$L = -\sum_{i} y_i \log(\hat{y}_i)$$Because $y$ is one-hot, every term is zero except the true class, and the whole thing collapses to:$$L = -\log(\hat{y}_{\text{true}})$$*The loss is just the negative log of the probability assigned to the correct answer.* That givesit exactly the shape you want:| Probability on correct class | Loss ||---|---|| 0.99 | 0.01 || 0.50 | 0.69 || 0.10 | 2.30 || 0.01 | 4.61 |Confident and right is nearly free. Confident and *wrong* is punished brutally — as$\hat{y}_{\text{true}} \to 0$, the loss goes to infinity. Cross-entropy does not merely want theright answer; it wants well-calibrated confidence.A useful sanity check: an untrained 3-class model should assign about ⅓ to everything, giving$-\log(1/3) \approx 1.0986$. If your initial loss is far from `ln(N)`, something is wrong beforetraining even starts.## The optimizerThe loss tells us how wrong we are; the optimizer decides what to do about it. We use **Adam**,which adapts a per-parameter learning rate from running estimates of the gradient's mean andvariance. It is the sensible default for small networks — far less fiddly than plain SGD.SKaiNET wires all three together with the `training { }` DSL:

In [11]:
import sk.ainet.lang.nn.loss.CategoricalCrossEntropyLoss
import sk.ainet.lang.nn.optim.adam
import sk.ainet.lang.nn.dsl.training

val runner = training<FP32, Float> {
    model { model }
    loss { CategoricalCrossEntropyLoss() }
    optimizer {
        adam(lr = 0.01).apply {
            model.trainableParameters().forEach { addParameter(it) }
        }
    }
}

println("Training runner ready: loss=${runner.loss::class.simpleName}, optimizer=${runner.optimizer::class.simpleName}")

Training runner ready: loss=CategoricalCrossEntropyLoss, optimizer=AdamOptimizer


`runner.step(ctx, x, y)` now performs a complete training step — forward pass, loss, backwardpass, optimizer update, gradient zeroing — in one call.> ### 🔧 Room for improvement>> Registering parameters with the optimizer requires reaching back into the model> (`model.trainableParameters().forEach { addParameter(it) }`) from inside the `optimizer { }`> block. Since the DSL already knows the model, this is a wiring step it could do itself —> `optimizer { adam(lr = 0.01) }` has all the information it needs. Forgetting the `.apply { }`> yields a model that trains silently and never improves, which is a nasty failure mode for a> beginner. Worth a follow-up issue.

---# 9. TrainingTraining is a loop over **epochs** (full passes over the training data). Within each epoch weprocess **mini-batches** — small groups of samples — rather than one sample at a time or the wholeset at once.Mini-batching is a compromise on two axes:- **Gradient quality.** A single sample gives a noisy gradient; the full dataset gives an exact  but expensive one. A batch of 16 averages out most of the noise cheaply.- **Speed.** Batched inputs become matrix–matrix multiplications, which vectorised CPU kernels  execute far more efficiently than a stream of matrix–vector products.A little noise is actually useful — it helps the optimizer escape poor local minima, which is whybatch training often *generalises better* than full-batch.SKaiNET's `Dataset.batches(batchSize, shuffle, seed)` returns a cold `Flow<DataBatch>`, and itreshuffles per epoch when you vary the seed. That matters: a fixed sample order lets the modellearn the *sequence* rather than the pattern.

In [12]:
import kotlinx.coroutines.flow.collect
import kotlinx.coroutines.runBlocking

val EPOCHS = 100
val BATCH_SIZE = 16

val history = mutableListOf<Pair<Int, Float>>()

runBlocking {
    repeat(EPOCHS) { epoch ->
        var epochLoss = 0.0f
        var batchCount = 0

        // New seed each epoch -> a different shuffle every pass.
        trainScaled.batches<FP32, Float>(
            batchSize = BATCH_SIZE,
            shuffle = true,
            seed = 1234L + epoch
        ).collect { batch ->
            // One call: forward, loss, backward, optimizer step, zero grad.
            val loss = runner.step(trainCtx, batch.x[0], batch.y)
            epochLoss += loss.data.get()
            batchCount++
        }

        val avgLoss = epochLoss / batchCount
        history += (epoch + 1) to avgLoss

        if (epoch == 0 || (epoch + 1) % 10 == 0) {
            println("Epoch ${(epoch + 1).toString().padStart(3)} / $EPOCHS   loss = ${"%.4f".format(avgLoss)}")
        }
    }
}

println()
println("Initial loss: ${"%.4f".format(history.first().second)}   (random guessing ~ ln(3) = 1.0986)")
println("Final loss:   ${"%.4f".format(history.last().second)}")

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[12], line 15, column 9: Unresolved reference: trainScaled
at Cell In[12], line 19, column 21: Cannot infer a type for this parameter. Please specify it explicitly.

The first epoch's loss should land near **1.0986** — that `ln(3)` sanity check from section 8,confirming the model starts out genuinely undecided. It should then fall steadily.Note what we did *not* write: no manual index arithmetic, no per-sample tensor construction, noshuffling code. The batching came from `Dataset`, and the entire train step from `runner.step`.> **Why the batch tensors work here.** Our `IrisDataset` builds tensors on a plain data context,> not the recording `trainCtx`. That is deliberate and supported: SKaiNET's cross-entropy> implementation routes the multiply through the *predictions'* ops rather than the targets', so> targets living on a non-recording context do not detach the gradient tape. Inputs and labels are> constants as far as autograd is concerned — only the weights need to be on the tape.

A quick text plot of the loss curve. The characteristic shape — a steep initial drop that flattensinto a long tail — is what healthy training looks like.

In [13]:
val sampled = history.filterIndexed { i, _ -> i % 5 == 0 || i == history.lastIndex }
val maxLoss = history.maxOf { it.second }

println("Training loss")
println("-".repeat(58))
sampled.forEach { (epoch, loss) ->
    val bars = ((loss / maxLoss) * 40).toInt().coerceAtLeast(1)
    println("${epoch.toString().padStart(3)} | ${"#".repeat(bars).padEnd(41)} ${"%.4f".format(loss)}")
}

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[13], line 1, column 15: Unresolved reference: history
at Cell In[13], line 1, column 39: Cannot infer a type for this parameter. Please specify it explicitly.
at Cell In[13], line 1, column 42: Cannot infer a type for this parameter. Please specify it explicitly.
at Cell In[13], line 1, column 66: Unresolved reference: history
at Cell In[13], line 2, column 15: Unresolved reference: history
at Cell In[13], line 2, column 31: Unresolved reference: it
at Cell In[13], line 6, column 9: Overload resolution ambiguity: 
public inline fun <T> Iterable<TypeVariable(T)>.forEach(action: (TypeVariable(T)) -> Unit): Unit defined in kotlin.collections
public inline fun <K, V> Map<out TypeVariable(K), TypeVariable(V)>.forEach(action: (Map.Entry<TypeVariable(K), TypeVariable(V)>) -> Unit): Unit defined in kotlin.collections
at Cell In[13], line 6, column 19: Cannot infer a type for this parameter. Please specify it explicitly.

---# 10. EvaluationNow we look at the test set — the 30 flowers the model has never seen.Two changes from training. First, we build an **evaluation context** with `Phase.EVAL` and nogradient tape: we are not learning, so recording operations would only waste memory. Second, weuse SKaiNET's built-in **`Accuracy`** metric rather than a hand-written argmax loop.`Accuracy` implements the `Metric` interface — the standard `update` / `compute` / `reset`accumulator pattern. You feed it batches, it tracks correct-versus-total, and it handles bothone-hot and integer targets, applying argmax where needed. Hand-rolling this is adozen lines of index juggling that can go subtly wrong.

In [14]:
import sk.ainet.lang.nn.metrics.Accuracy

val evalCtx = DefaultGraphExecutionContext(
    baseOps = baseCtx.ops,
    phase = Phase.EVAL
)

fun evaluate(data: IrisDataset, label: String): Double {
    val accuracy = Accuracy()

    runBlocking {
        data.batches<FP32, Float>(batchSize = 32, shuffle = false).collect { batch ->
            val logits = model.forward(batch.x[0], evalCtx)
            accuracy.update(logits, batch.y, evalCtx)
        }
    }

    val score = accuracy.compute()
    println("${label.padEnd(18)} ${"%.2f".format(score * 100)}%")
    return score
}

val trainAcc = evaluate(trainScaled, "Train accuracy:")
val testAcc = evaluate(testScaled, "Test accuracy:")

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[14], line 11, column 5: Unresolved reference: runBlocking
at Cell In[14], line 12, column 68: Suspend function 'collect' should be called only from a coroutine or another suspend function
at Cell In[14], line 23, column 25: Unresolved reference: trainScaled
at Cell In[14], line 24, column 24: Unresolved reference: testScaled

Compare the two numbers. Similar scores mean the model generalised. A high train score with amarkedly lower test score means **overfitting** — memorisation rather than learning. On Iris witheight hidden units there is not enough capacity to memorise much, so they should be close.## Beyond accuracy: the confusion matrixA single accuracy number hides *how* a model fails, and the failure pattern is usually theinteresting part. A **confusion matrix** shows every combination of true and predicted class:rows are the truth, columns the prediction, and the diagonal holds the correct answers.For Iris the classic result is that *setosa* is never confused with anything, while *versicolor*and *virginica* occasionally trade places — they genuinely overlap in petal measurements. Accuracyalone would not tell you that; the matrix does, and it points straight at which feature toinvestigate.

In [15]:
fun confusionMatrix(data: IrisDataset): Array<IntArray> {
    val matrix = Array(NUM_CLASSES) { IntArray(NUM_CLASSES) }

    runBlocking {
        data.batches<FP32, Float>(batchSize = 32, shuffle = false).collect { batch ->
            val logits = model.forward(batch.x[0], evalCtx)
            for (row in 0 until batch.batchSize) {
                val predicted = (0 until NUM_CLASSES).maxBy { c -> logits.data[row, c] }
                val actual = (0 until NUM_CLASSES).maxBy { c -> batch.y.data[row, c] }
                matrix[actual][predicted]++
            }
        }
    }
    return matrix
}

val cm = confusionMatrix(testScaled)

val shortNames = CLASS_NAMES.map { it.removePrefix("Iris-").take(10) }

println("Confusion matrix (test set) — rows = actual, columns = predicted")
println()
println(" ".repeat(13) + shortNames.joinToString("") { it.padStart(12) })
cm.forEachIndexed { actual, row ->
    println(shortNames[actual].padEnd(13) + row.joinToString("") { it.toString().padStart(12) })
}
println()
CLASS_NAMES.indices.forEach { c ->
    val total = cm[c].sum()
    val correct = cm[c][c]
    if (total > 0) {
        println("${shortNames[c].padEnd(12)} recall = $correct/$total  (${"%.1f".format(100.0 * correct / total)}%)")
    }
}

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[15], line 4, column 5: Unresolved reference: runBlocking
at Cell In[15], line 5, column 68: Suspend function 'collect' should be called only from a coroutine or another suspend function
at Cell In[15], line 6, column 52: Unresolved reference: evalCtx
at Cell In[15], line 17, column 26: Unresolved reference: testScaled

---# 11. Making a predictionFinally, the point of the exercise: classify a flower.Three steps, and each mirrors something from training.1. **Standardise** with the scaler fitted on the training set — the same μ and σ, never recomputed.   Skipping this is a classic production bug: the model sees raw centimetres when it was trained on   z-scores, and predicts nonsense.2. **Forward pass** to get logits.3. **Softmax** — and *here* is where it finally belongs. During training the loss applied it   internally; now we want human-readable probabilities, so we apply it ourselves.

In [16]:
import sk.ainet.lang.tensor.softmax

fun predict(sepalLength: Float, sepalWidth: Float, petalLength: Float, petalWidth: Float) {
    val raw = floatArrayOf(sepalLength, sepalWidth, petalLength, petalWidth)

    // 1. Standardise with the TRAINING statistics.
    val scaled = scaler.apply(raw)

    // 2. Forward pass -> logits. Shape [1, 3]: a batch of one.
    val input = baseCtx.fromFloatArray<FP32, Float>(Shape(1, NUM_FEATURES), FP32::class, scaled)
    val logits = model.forward(input, evalCtx)

    // 3. Softmax -> probabilities, for humans.
    val probs = logits.softmax(dim = -1)

    val predicted = (0 until NUM_CLASSES).maxBy { probs.data[0, it] }

    println("Measurements: sepal ${sepalLength}x${sepalWidth} cm, petal ${petalLength}x${petalWidth} cm")
    println("  -> ${CLASS_NAMES[predicted]}  (${"%.1f".format(probs.data[0, predicted] * 100)}% confident)")
    println()
    CLASS_NAMES.indices.forEach { c ->
        val p = probs.data[0, c]
        val bar = "#".repeat((p * 30).toInt())
        println("     ${shortNames[c].padEnd(12)} ${"%.4f".format(p)} $bar")
    }
    println()
}

// A textbook setosa: tiny petals.
predict(5.1f, 3.5f, 1.4f, 0.2f)

// A textbook virginica: large petals.
predict(6.3f, 3.3f, 6.0f, 2.5f)

// Deliberately ambiguous — sits in the versicolor/virginica overlap.
predict(6.0f, 2.7f, 5.1f, 1.6f)

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[16], line 7, column 18: Unresolved reference: scaler
at Cell In[16], line 11, column 39: Unresolved reference: evalCtx
at Cell In[16], line 24, column 25: Unresolved reference: shortNames

The third example is the instructive one. A well-trained multiclass model does not just pick awinner; it expresses *uncertainty* about genuinely ambiguous input by spreading probabilityacross the plausible classes. That is the softmax distribution doing real work — and it isinformation you would throw away if you only ever looked at `argmax`.

---# 12. Recap, and what SKaiNET did for us## The concepts1. **Multiclass ≠ multi-label.** Mutually exclusive classes are what justify softmax.2. **One-hot encode labels** so no false ordering is implied.3. **Stratify your splits** to keep class balance in both halves.4. **Fit preprocessing on the training set only** — statistics from test data are leakage.5. **Output logits during training, softmax at inference.** Double-softmaxing produces a model   that looks fine and trains badly.6. **Cross-entropy is $-\log(\text{p of the correct class})$** — it rewards calibration, not just   correctness. Check your initial loss against `ln(N)`.7. **Look past accuracy.** A confusion matrix shows *which* classes the model conflates.## The engineMost of this notebook is prose, and that is the point — the code stayed short because SKaiNETsupplied the machinery:| Job | What we used ||---|---|| Architecture | `sequential<FP32, Float> { }` with in-layer `activation` and shape-aware `weights { }` || Split | `Dataset.split(ratio, seed, stratified = true)` || Batching | `Dataset.batches(...)` as a `Flow<DataBatch>` || Train step | `training { }` DSL + `runner.step(...)` || Evaluation | `Accuracy` metric with `update` / `compute` / `reset` || Autograd | `DefaultGraphExecutionContext` + `DefaultGradientTape` |Type safety is worth a mention too. `Tensor<FP32, Float>` carries its precision in the typesystem, so a dtype mismatch is a compile error rather than a runtime surprise three hours into atraining run. And because this is plain Kotlin on the JVM, the model drops straight into a Springservice or an Android app — no Python runtime, no serving bridge, no cross-language glue.## The gapSection 4 was the sore thumb: forty lines of hand-written `Dataset` plumbing to load a CSV thatSKaiNET can already *parse*. The companion issue proposes closing that gap with a tabular`RawDataset → Dataset` adapter plus a first-class `Iris` provider, so this notebook's dataloading becomes a single line.## Where to go next- **Tune it.** Change the hidden layer width, swap `adam` for `sgd(lr = 0.1, momentum = 0.9)`,  vary the batch size. Watch the loss curve respond.- **Break it deliberately.** Add `activation = { it.softmax() }` to the output layer and compare  the loss curves. Seeing the double-softmax slowdown first-hand makes the lesson stick.- **Skip standardisation** and count how many more epochs it takes to converge.- **Go bigger.** [`MNISTDemo`](https://github.com/SKaiNET-developers/SKaiNET-examples) applies the  same pattern to 60,000 handwritten digits — ten classes instead of three, and a real dataset  provider instead of section 4's boilerplate.## References- [SKaiNET](https://github.com/SKaiNET-developers/SKaiNET) — core engine- [SKaiNET-examples](https://github.com/SKaiNET-developers/SKaiNET-examples) — runnable samples, including the `IrisDemo` this notebook grew from- [SKaiNET-notebook](https://github.com/SKaiNET-developers/SKaiNET-notebook) — more interactive notebooks- Fisher, R.A. (1936), *The Use of Multiple Measurements in Taxonomic Problems*